In [1]:
from utils import *
from param_search_var_osc import *

0.2s / iter

Tot iter: 50*140*140*140 = 137,200,200 iter = 27,440,000 s = 2744 s / job = 45 min /job

In [23]:
EJ_subdivisions = 50
EC_subdivisions = 140
EL_subdivisions = 140
Er_subdivisions = 140

EJ_values = np.linspace(4, 16, EJ_subdivisions)
EC_values = np.linspace(0.1, 8, EC_subdivisions)
EL_values = np.linspace(0.1, 1, EL_subdivisions)
Er_values = np.linspace(3, 18, Er_subdivisions)
EC_EL_extents = [EC_values[0], EC_values[-1], EL_values[0], EL_values[-1]]

total_jobs = 10000
existing_chunk_num = 0
num_chunks_per_EJ = total_jobs // len(EJ_values) # = 20

# num_done = 0
# for EJ in EJ_values:
#     EC_grid, EL_grid, Er_grid = np.meshgrid(EC_values, EL_values, Er_values)
#     EC_flat = EC_grid.flatten()
#     EL_flat = EL_grid.flatten()
#     Er_flat = Er_grid.flatten()
#     EC_chunks = np.array_split(EC_flat, num_chunks_per_EJ)
#     EL_chunks = np.array_split(EL_flat, num_chunks_per_EJ)
#     Er_chunks = np.array_split(Er_flat, num_chunks_per_EJ)

#     for EC_chunk, EL_chunk,Er_chunk in zip(EC_chunks, EL_chunks, Er_chunks):
#         job = search_job(EJ, EC_chunk, EL_chunk,Er_chunk)
#         with open(f'{existing_chunk_num}.pkl', 'wb') as f:
#             pickle.dump(job, f)
#         existing_chunk_num += 1
#     num_done+=1
#     clear_output()
#     print(f"num done:{num_done}/50")


# def pack_pkl_files_to_zip(zip_filename="param_search.zip"):
#     # Create a new ZIP file
#     print('zipping')
#     with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
#         # Loop through all files in the current directory
#         for filename in os.listdir('.'):
#             # Check if the file is a .pkl file with an integer name
#             name, ext = os.path.splitext(filename)
#             if ext == '.pkl' and name.isdigit():
#                 # Add the file to the ZIP
#                 zipf.write(filename)
#                 # Delete the .pkl file
#                 os.remove(filename)
#     print('zipping done')
                
# pack_pkl_files_to_zip()


In [24]:

one_two_grid = np.zeros((EJ_subdivisions, EC_subdivisions, EL_subdivisions, Er_subdivisions))
differential_stark_grid = np.zeros((EJ_subdivisions, EC_subdivisions, EL_subdivisions, Er_subdivisions))
qubit_zero_lamb_grid = np.zeros((EJ_subdivisions, EC_subdivisions, EL_subdivisions, Er_subdivisions))
detunning_grid = np.zeros((EJ_subdivisions, EC_subdivisions, EL_subdivisions, Er_subdivisions))

# Initialize counter for existing chunks
existing_chunk_num = 0
# Loop through each EJ value
chunk_len = int(EC_subdivisions*EL_subdivisions * Er_subdivisions / num_chunks_per_EJ)
for EJ_idx, EJ in enumerate(EJ_values):    
    # Initialize flattened arrays to store the results for this EJ value
    total_elements = EC_subdivisions * EL_subdivisions* Er_subdivisions

    transition_flat = np.zeros(total_elements)
    zero_three_flat = np.zeros(total_elements)
    sum_of_differential_stark_on_qubit_12_flat = np.zeros(total_elements)
    sum_of_qubit_zero_lamb_on_osc_flat = np.zeros(total_elements)
    detunning_qubit01_flat = np.zeros(total_elements)
    
    flat_idx = 0
    # Read that many chunks for this EJ value
    for _ in range(total_jobs // len(EJ_values)):
        try:
            with gzip.GzipFile(f"param_search_result_osc_v2/result_{existing_chunk_num}.zip", "rb") as f:
                job = pickle.load(f)
            
            # Assuming job.results is a tuple of 4 numpy arrays, each of shape (len(job.EC_values), len(job.EL_values))
            transition,sum_of_differential_stark_on_qubit_12, sum_of_qubit_zero_lamb_on_osc, detunning_qubit01 = job.results
                        
            # Place the results back into the flattened arrays
            transition_flat[flat_idx:flat_idx + chunk_len] = transition
            sum_of_differential_stark_on_qubit_12_flat[flat_idx:flat_idx + chunk_len] = sum_of_differential_stark_on_qubit_12
            sum_of_qubit_zero_lamb_on_osc_flat[flat_idx:flat_idx + chunk_len] = sum_of_qubit_zero_lamb_on_osc
            detunning_qubit01_flat[flat_idx:flat_idx + chunk_len] = detunning_qubit01
            
            # Update the index for the flattened arrays
            flat_idx += chunk_len
        except Exception as e:
            clear_output()
            print(f"Error occurred while loading chunk {existing_chunk_num}: {e}")
            transition_flat[flat_idx:flat_idx + chunk_len] = np.full(chunk_len, None)
            sum_of_differential_stark_on_qubit_12_flat[flat_idx:flat_idx + chunk_len] = np.full(chunk_len, None)
            sum_of_qubit_zero_lamb_on_osc_flat[flat_idx:flat_idx + chunk_len] = np.full(chunk_len, None)
            detunning_qubit01_flat[flat_idx:flat_idx + chunk_len] = np.full(chunk_len, None)

            # Update the index for the flattened arrays
            flat_idx += chunk_len 
        existing_chunk_num += 1
    
    # Reshape the flattened arrays back into the original grid for this EJ value
    one_two_grid[EJ_idx, :, :, :] = transition_flat.reshape(EC_subdivisions, EL_subdivisions, Er_subdivisions)
    differential_stark_grid[EJ_idx, :, :, :] = sum_of_differential_stark_on_qubit_12_flat.reshape(EC_subdivisions, EL_subdivisions, Er_subdivisions)
    qubit_zero_lamb_grid[EJ_idx, :, :, :] = sum_of_qubit_zero_lamb_on_osc_flat.reshape(EC_subdivisions, EL_subdivisions, Er_subdivisions)
    detunning_grid[EJ_idx, :, :, :] = detunning_qubit01_flat.reshape(EC_subdivisions, EL_subdivisions, Er_subdivisions)


In [25]:
from matplotlib.colors import LogNorm
from ipywidgets import interact, FloatSlider
import matplotlib.pyplot as plt
import numpy as np


def plot_all(EJ, Er):
    EJ_idx = np.argmin(np.abs(EJ_values - EJ))
    Er_idx = np.argmin(np.abs(Er_values - Er))
    EJ = EJ_values[EJ_idx]
    Er = Er_values[Er_idx]
    plt.figure(figsize=(15, 10))
    
    plt.subplot(2, 2, 1)
    plt.imshow(one_two_grid[EJ_idx, :, :, Er_idx], extent=EC_EL_extents, origin='lower', aspect='auto', norm=LogNorm(vmax = 5e-1,vmin = 1e-4))
    plt.colorbar(label='')
    plt.xlabel('EC')
    plt.ylabel('EL')
    plt.title(f'one-two transition for EJ = {EJ}, Er = {Er}')
    
    plt.subplot(2, 2, 2)
    plt.imshow(detunning_grid[EJ_idx, :, :, Er_idx], extent=EC_EL_extents, origin='lower', aspect='auto', norm=LogNorm(vmax = 1e-1,vmin = 1e-3))
    plt.colorbar(label='')
    plt.xlabel('EC')
    plt.ylabel('EL')
    plt.title(f'detunning: min(abs(00->01 - 10->11),abs(00->01 - 20->21))')
    
    plt.subplot(2, 2, 3)
    plt.imshow(qubit_zero_lamb_grid[EJ_idx, :, :, Er_idx], extent=EC_EL_extents, origin='lower', aspect='auto', norm=LogNorm(vmax = 1e-1,vmin = 1e-4))
    plt.colorbar(label='')
    plt.xlabel('EC')
    plt.ylabel('EL')
    plt.title(f'qubit_0_differential_lamb: max(abs(00->01 - 10->11),abs(00->01 - 20->21))')
    
    plt.subplot(2, 2, 4)
    plt.imshow(differential_stark_grid[EJ_idx, :, :, Er_idx], extent=EC_EL_extents, origin='lower', aspect='auto', norm=LogNorm(vmax = 1e-1,vmin = 1e-4))
    plt.colorbar(label='')
    plt.xlabel('EC')
    plt.ylabel('EL')
    plt.title(f'differential_stark_on_qubit_12:  max(abs(10->20 - 11->21),abs(11->21 - 12->22))')


    plt.tight_layout()
    plt.show()

interact(plot_all, 
         EJ=FloatSlider(min=EJ_values[0], max=EJ_values[-1], step=(EJ_values[1]-EJ_values[0]), value=EJ_values[0]),
         Er=FloatSlider(min=Er_values[0], max=Er_values[-1], step=(Er_values[1]-Er_values[0]), value=Er_values[0]))


interactive(children=(FloatSlider(value=4.0, description='EJ', max=16.0, min=4.0, step=0.24489795918367374), F…

<function __main__.plot_all(EJ, Er)>

# End of the notebook